# Churn Risk Drivers Analysis

In [1]:
import os
import sys
from pathlib import Path

# Get project root (notebooks/ -> use_case/)
project_root = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
# IMPORTANT: Update this path to point to your use_case folder if you have any issue
# For example: project_root = Path(r"C:\path\to\your\use_case")
sys.path.append(str(project_root))
os.chdir(project_root)

In [2]:
from scripts.data_cleaner import drop_negative_values, fill_null_with_average, fill_null_with_mode, fill_os_with_device_model, fill_null_with_random_bernouilli, remove_duplicates
from scripts.preprocessor import age_segmentation, session_length_segmentation, prepare_features
from scripts.visualizer import qualitative_variables_impacts_on_retention, quantitative_variable_impact_on_retention, correlation_study, retention_comparison, roc_curves_comparison, retention_correlation_heatmap
from scripts.formatter import export_to_pdf
from scripts.statistical_analyzer import chi2_test_multiple, retention_correlation_matrix
from scripts.feature_importance import compute_feature_importance
from sklearn.model_selection import train_test_split
import pandas as pd
import os

In [3]:
# Load all data
data_path = project_root / "ressource"
onboarding_data = pd.read_csv(data_path / "d0_behaviour_br.csv", delimiter=';')
retention_data = pd.read_csv(data_path / "retention_br.csv", delimiter=';')

In [4]:
# Clean onboarding dataset
retention_columns = ['d0', 'd3', 'd7', 'd14', 'd30']
onboarding_data_cleaned = (onboarding_data.copy()
    .pipe(remove_duplicates)
    .pipe(drop_negative_values, 'session_length')
    .pipe(fill_null_with_average, 'age')
    .pipe(fill_null_with_average, 'session_length')
    .pipe(fill_null_with_mode, 'country')
    .pipe(fill_null_with_mode, 'locale')
    .pipe(fill_null_with_mode, 'device_model')
    .pipe(fill_os_with_device_model)
    .pipe(fill_null_with_random_bernouilli, 'had_meaningful'))


# Clean retention dataset
retention_data_cleaned = retention_data.copy()
retention_data_cleaned[retention_columns] = retention_data_cleaned[retention_columns].fillna(0)


-- Removing duplicates
Initial count: 112770
Dropped 4 duplicate row(s)
Final count: 112766

-- Managing negative outliers for column: session_length
Dropped 1 rows

-- Managing nulls for quantitative column: age
Number of null values: 30822
Filled with average value: 17

-- Managing nulls for quantitative column: session_length
Number of null values: 10775
Filled with average value: 2134

-- Managing nulls for qualitative column: country
Number of null values: 2
Filled nulls with most frequent value: US

-- Managing nulls for qualitative column: locale
Number of null values: 13
Filled nulls with most frequent value: en-US

-- Managing nulls for qualitative column: device_model
Number of null values: 1
Filled nulls with most frequent value: iPhone12,1

-- Managing nulls for OS column using device_model
Number of null values: 6
Replaced 6 null(s) with iOS (found iPhone or iPad in device_model)
Replaced 0 null(s) with Android (the rest)

-- Managing nulls for binary column: had_meaningf

In [5]:
# Preprocess: merge datasets and add segmented columns
merged_dataset = (onboarding_data_cleaned
    .merge(retention_data_cleaned, on='keychain_udid', how='left')
    .pipe(lambda df: df.assign(**{col: df[col].fillna(0) for col in retention_columns}))
    .pipe(age_segmentation)
    .pipe(session_length_segmentation))


total_nulls = merged_dataset.isnull().sum().sum()
print(f"\n-- Quality test to check for remaining null")
print(f"✓ No nulls were found in this dataset" if total_nulls == 0 else f"✗ Found {total_nulls} null(s) in this dataset")


-- Quality test to check for remaining null
✗ Found 12 null(s) in this dataset


In [6]:
# Retention correlation heatmap
retention_corr_fig = retention_correlation_heatmap(merged_dataset)

In [7]:
# Check overlap between d14 and d30
d30_count = (merged_dataset['d30'] == 1).sum()
both_count = ((merged_dataset['d14'] == 1) & (merged_dataset['d30'] == 1)).sum()

print(f"Of users with d30 = 1, percentage who also have d14 = 1: {both_count/d30_count*100:.2f}%" if d30_count > 0 else "")

Of users with d30 = 1, percentage who also have d14 = 1: 60.19%


## Retention target variables correlation

The analysis shows the overlap between users retained at day 14 (d14) and day 30 (d30). 

**Key Finding:** 
- Globally, only 60% of people who are active after 30 days were already active at day 14. 
- This indicates that a significant portion of d30 active users (40%) were not active at d14, which helps explain the lower correlation between these retention periods. 
- Correlation matrix shows medium correlation between all variables

> **Conclusion:** It's not because you come back earlier that you create long-term retention. So for the business point of view we want to focus on **d30 as the retention KPI to optimise**.

In [8]:
# Visualizations
retention_fig = retention_comparison(merged_dataset)

# Define target variable (can be changed to 'd7', 'd3', etc.)
target_variable = 'd30'
qual_figs = qualitative_variables_impacts_on_retention(merged_dataset, ['country', 'os', 'locale', 'device_model','had_meaningful'], target_variable)

In [9]:
quant_figs = quantitative_variable_impact_on_retention(merged_dataset, target_var=target_variable)

In [10]:
# Chi-square tests table
chi2_results = chi2_test_multiple(merged_dataset, ['age_segmented', 'session_length_segmented', 'had_meaningful', 'os', 'country'], target_variable)
chi2_results['Is Significant'] = (chi2_results['Chi2 P-Value'] < 0.05).map({True: 'Yes', False: 'No'})
chi2_results

,Variable,Chi2 P-Value,Is Significant
0,age_segmented,8.394776e-286,Yes
1,session_length_segmented,0.000000e+00,Yes
2,had_meaningful,4.577400e-143,Yes
3,os,9.498093e-199,Yes
4,country,2.485265e-45,Yes


## Churn drivers identification

There is a clear correlation and dependency between the target variable (d30) and the study variables (age, session_length, had_meaningful, os, country). 

This dependency is evident in two ways:

1. **Visual evidence from the graphics**: The visualizations show a non-uniform, non-random pattern. When one variable changes, the target variable (d30) clearly moves in response - we can see systematic variations rather than random fluctuations.

2. **Statistical proof through Chi-square tests**: The Chi-square test results prove this dependency is statistically significant (all p-values < 0.05). This confirms that the relationships observed in the graphics are not due to chance, but represent genuine associations between the features and retention at day 30.


In [11]:
# Correlation study: heatmaps
correlation_figs = correlation_study(merged_dataset)

## Churn Drivers Correlations each others

We observed that all variables are globally correlated with our study variable (d30). However, for the feature importance analysis, we will keep only the following variables:

- **Segmented variables**: session_length_segmented, age_segmented
- **had_meaningful**
- **os**
- **country**

### Excluded Variables:

1. **Locale and Timezone**: We found that country is too correlated with locale and timezone, as we can see in the graphics and confirmed by the Chi-square test. The variables are significantly correlated between them, especially between country-locale and country-timezone, which is logical. Therefore, we keep only **country** to avoid redundancy.

2. **Device Model**: We keep only **OS** and not device_model. The reason is that device_model cuts the sample too much. We can already see in the graphs before that there was too much gray, where gray represents the fact that there are less than 300 observations - making these segments not viable for analysis.

### Correlation Considerations:

3. **Session Length and Had Meaningful**: We do not consider the correlation between session_length and had_meaningful, even though there is a statistically significant correlation. While the data shows a relationship, it is not meaningful for our analysis. The pattern is not distinct - I can't see a logical relationship like for instance  the more meaningful reactions we have, the longer the sessions tend to be. so we are not comfortable keeping this correlation in our considerations.


In [12]:
# Feature importance using multiple models
variables_to_test = ['country', 'os', 'age', 'session_length','had_meaningful']

# Prepare features (encode categorical variables)
df_prepared = prepare_features(merged_dataset, variables_to_test, target_var=target_variable)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df_prepared[variables_to_test], 
    df_prepared[target_variable], 
    test_size=0.2, 
    random_state=83
)

# Compute feature importance
importance_results, roc_data, best_model, best_score = compute_feature_importance(
    X_train, X_test, y_train, y_test, variables_to_test
)
importance_results 


Best Model: XGBoost
ROC AUC Score: 0.7005


,Feature,Importance
0,session_length,0.311481
1,os,0.267742
2,age,0.209222
3,had_meaningful,0.112403
4,country,0.099151


In [13]:
# Create ROC curve graph
roc_fig = roc_curves_comparison(roc_data, best_model.name, best_score)

## Churn Drivers Importance

Overall, we used a data science model only in order to quantify the weight and strength of each of the churn drivers that we retained post analytics. 
We observe a ROC of 0.7 which is correct, so we can rely on this ranking and what comes out of it.

> The main drivers is **session_length**


### Global Conclusions of the Whole Study:

> **Session Length and Had Meaningful** are for me the two most important drivers because we can really interact on them.

> **Age and Country** is more complicated. We could act on it. We shall explore profiling of our clients to try to personalize the content to make them fit better, if we assume that it is the same application for all countries.

> **OS - Important Distinction**: We still observe 5 points of retention less on Android. In this context, there is clearly potentially a bottleneck in the user experience that would need work. 

In [14]:
# Export all graphs to PDF (including retention comparison, heatmaps, and ROC curve)
all_figs = [retention_fig] + [retention_corr_fig] + qual_figs + quant_figs + correlation_figs + [roc_fig]
export_to_pdf(all_figs)


Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an alpha channel. Computing a separate soft mask (/SMask) image to store transparency in PDF.
Image contains an a

PDF exported to: C:\Users\as_cu\Desktop\use_case\output\churn_risk_analysis.pdf
